In [1]:
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
BiocManager::install("BiocParallel")
BiocManager::install("limma")
BiocManager::install("BiocGenerics")
install.packages("config")

library(tidyverse)
library(stringr)
library(tibble)
library(config)
library(limma)
library(BiocGenerics)

'getOption("repos")' replaces Bioconductor standard repositories, see
'?repositories' for details

replacement repositories:
    CRAN: https://cran.r-project.org

Bioconductor version 3.10 (BiocManager 1.30.16), R 3.6.1 (2019-07-05)
Warning message:
"package(s) not installed when version(s) same as current; use `force = TRUE` to
  re-install: 'BiocParallel'"Old packages: 'float', 'askpass', 'backports', 'BH', 'boot', 'brio', 'broom',
  'cachem', 'caret', 'class', 'cli', 'clipr', 'cluster', 'codetools',
  'colorspace', 'cpp11', 'crayon', 'credentials', 'curl', 'data.table', 'DBI',
  'dbplyr', 'diffobj', 'digest', 'dplyr', 'evaluate', 'fansi', 'forcats',
  'foreach', 'formatR', 'fs', 'generics', 'gert', 'ggplot2', 'glmnet', 'glue',
  'gower', 'haven', 'hexbin', 'highr', 'hms', 'htmltools', 'htmlwidgets',
  'httpuv', 'ipred', 'IRdisplay', 'IRkernel', 'iterators', 'jsonlite',
  'KernSmooth', 'knitr', 'labeling', 'later', 'lattice', 'lava', 'lubridate',
  'magrittr', 'maps', 'markdown', 'MA

package 'config' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\trungn\AppData\Local\Temp\RtmpkDMbpr\downloaded_packages


Registered S3 methods overwritten by 'ggplot2':
  method         from 
  [.quosures     rlang
  c.quosures     rlang
  print.quosures rlang
Registered S3 method overwritten by 'rvest':
  method            from
  read_xml.response xml2
-- Attaching packages ------------------------------------------------------------------------------- tidyverse 1.2.1 --
v ggplot2 3.1.1       v purrr   0.3.2  
v tibble  2.1.1       v dplyr   0.8.0.1
v tidyr   0.8.3       v stringr 1.4.0  
v readr   1.3.1       v forcats 0.4.0  
-- Conflicts ---------------------------------------------------------------------------------- tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
Warning message:
"package 'config' was built under R version 3.6.3"
Attaching package: 'config'

The following objects are masked from 'package:base':

    get, merge

Warning message:
"package 'limma' was built under R version 3.6.2"Loading required package: parallel

Attaching packag

In [2]:
dbSafeNames = function(names) {
  names = gsub('[^a-z0-9]+','_',str_to_lower(names))
  names = make.names(names, unique=TRUE, allow_=TRUE)
  names = gsub('.','_',names, fixed=TRUE)
  names
}

checkFilePath = function(path){
    return(file.exists(path))}

tryReadTXTFile= function(file){
    tryRead <- try(read.table(file, sep = "", header = FALSE), silent = TRUE)
    if (class(tryRead) != "try-error") {
        name <- read.table(file, sep = "", header = FALSE)
    }else{
        message("File doesn't exist, please check")
    }

    return(name)
}


tryReadTSVFile = function(file){
    tryRead <- try(read_tsv(file), silent = TRUE)
    if (class(tryRead) != "try-error") {
        name <- read_tsv(file)
    } else {
        message("File doesn't exist, please check")
    }
    return(name)
}

In [3]:
microarray_counts_path = "C:/Users/trungn/PycharmProjects/DGEAP/validation/microarray_endometriosis_counts.tsv"
microarray_col_path = "C:/Users/trungn/PycharmProjects/DGEAP/validation/microarray_endometriosis_coldata.tsv"
input_config_path = "C:/Users/trungn/PycharmProjects/DGEAP/validation/config.yml"
filter_gene_path = "C:/Users/trungn/PycharmProjects/DGEAP/validation/filter_gene.txt"

In [4]:
if (checkFilePath(microarray_counts_path) &
    checkFilePath(microarray_col_path) &
    checkFilePath(input_config_path)
    ){
        print("a")
        counts_df <- tryReadTSVFile(microarray_counts_path)
        print("b")
        coldata_df <- tryReadTSVFile(microarray_col_path)
        print("c")
        input_config<- input_config<- config::get(
            config = Sys.getenv("R_CONFIG_ACTIVE", "microarray_analysis"),
            file = Sys.getenv("R_CONFIG_FILE", input_config_path),
            use_parent = TRUE)
    print(2)
    # get the parameters from user
    a <- input_config$microarray_analysis$min_expr_log_base_b
    b <- input_config$microarray_analysis$min_expr_lob_base_b_of_a
    min_expr <- log(b,a)
    print("min expr")
    print(min_expr)
    min_prop <- input_config$microarray_analysis$min_prop
    # padj_thresh <- microarray_config$padj_thresh
    condition <- input_config$microarray_analysis$condition
    adj_method <- input_config$microarray_analysis$adj_method
    use_weight <- input_config$microarray_analysis$use_weight
    contrast_level <- input_config$microarray_analysis$contrast_level
    reference_level <- input_config$microarray_analysis$reference_level
    print(3)
    print(min_expr)
    print(min_prop)
    print(condition)
    print(adj_method)
    print(typeof(use_weight))
    if (nrow(counts_df) > 1 & nrow(coldata_df) > 1){
        print(4)
        coldata_df <- coldata_df %>%
        mutate(condition = factor(condition, levels = c(contrast_level, reference_level)))
        print(5)
        coef_name = gsub(" ", "", paste(condition, reference_level))
        print(coef_name)

        filt_counts_df <- counts_df %>%
            dplyr::filter(rowSums(.[-1] > min_expr) / (ncol(.) - 1) >= min_prop)
        print(6)
        filt_expr <- filt_counts_df %>%
            column_to_rownames("symbol") %>%
            as.matrix()
        print(7)
        design <- model.matrix(~ condition, data = coldata_df)
        rownames(design) <- coldata_df$sample_name
        design
        all(colnames(filt_expr) == rownames(design))
        print(9)
        if (isTRUE(use_weight)){
            qual_weights <- arrayWeights(filt_expr, design = design)
            print(10)
        }else{
            qual_weights <- 0
        }
        print(11)
        lm_fit <- lmFit(filt_expr, design = design, weights = qual_weights)
        print(12)
        bayes_fit <- eBayes(lm_fit)
        print(13)
        bayes_fit$coefficients %>% colnames()
        print(14)
        fit_de_res_df_unfiltered_genes <- topTable(bayes_fit, coef = coef_name, number = nrow(filt_counts_df), adjust.method = adj_method) %>%
            rename(lfc = logFC, ave_expr = AveExpr, pval = P.Value, padj = adj.P.Val) %>%
           as_tibble(rownames = "symbol")
        print(15)
        # save the data table output as tsv
        write.table(fit_de_res_df_unfiltered_genes,
            file='microrray_result_unfiltered.tsv', quote=FALSE, sep='\t', col.names = NA)
        print(16)


        if (checkFilePath(filter_gene_path)){
            filter_gene_df <- tryReadTXTFile(filter_gene_path)
            if(nrow(filter_gene_df) > 1){
                print("start")
                gene_vec <- filter_gene_df[['V1']]
                print(gene_vec)
                print(18)
                modified_counts_df <- counts_df[counts_df$symbol %in% gene_vec,]
                print(modified_counts_df)

                modified_filt_counts_df <- modified_counts_df %>%
                    dplyr::filter(rowSums(.[-1] > min_expr) / (ncol(.) - 1) >= min_prop)
                print(19)
                modified_filt_expr <- modified_filt_counts_df %>%
                    column_to_rownames("symbol") %>%
                    as.matrix()
                print('==')

                design <- model.matrix(~ condition, data = coldata_df)
                rownames(design) <- coldata_df$sample_name


                design
                all(colnames(modified_filt_expr) == rownames(design))
                print(21)
                print(design)
                if (isTRUE(use_weight)){
                    print("===")
                    modified_qual_weights <- arrayWeights(modified_filt_expr, design = design)
                    print("=====")
                print(10)
                }else{
                    modified_qual_weights <- 0
                }
                print(22)
                modified_lm_fit <- lmFit(modified_filt_expr, design = design, weights = modified_qual_weights)
                print(23)
                modified_bayes_fit <- eBayes(modified_lm_fit)
                modified_bayes_fit$coefficients %>% colnames()
                print("====")
                fit_de_res_df_filtered_genes <- topTable(modified_bayes_fit, coef = coef_name, number = nrow(modified_filt_counts_df), adjust.method = "BH") %>%
                rename(lfc = logFC, ave_expr = AveExpr, pval = P.Value, padj = adj.P.Val) %>%
                as_tibble(rownames = "symbol")
                print("======")
                colnames(fit_de_res_df_filtered_genes) = dbSafeNames(colnames(fit_de_res_df_filtered_genes))
                print(24)
                # save the data table output as tsv
                write.table(fit_de_res_df_filtered_genes,
                    file='microrray_result_filtered.tsv', quote=FALSE, sep='\t', col.names = NA)
                print(25)
            }
        }
    }

}

[1] "a"


Parsed with column specification:
cols(
  .default = col_double(),
  symbol = col_character()
)
See spec(...) for full column specifications.
Warning message in if (class(tryRead) != "try-error") {:
"the condition has length > 1 and only the first element will be used"Parsed with column specification:
cols(
  .default = col_double(),
  symbol = col_character()
)
See spec(...) for full column specifications.


[1] "b"


Parsed with column specification:
cols(
  sample_name = col_character(),
  series = col_character(),
  title = col_character(),
  condition = col_character(),
  endometriosis_stage = col_character(),
  phase = col_character(),
  tissue = col_character(),
  batch = col_double()
)
Warning message in if (class(tryRead) != "try-error") {:
"the condition has length > 1 and only the first element will be used"Parsed with column specification:
cols(
  sample_name = col_character(),
  series = col_character(),
  title = col_character(),
  condition = col_character(),
  endometriosis_stage = col_character(),
  phase = col_character(),
  tissue = col_character(),
  batch = col_double()
)


[1] "c"


Warning message in readLines(con):
"incomplete final line found on 'C:\Users\trungn\PycharmProjects\DGEAP\validation\config.yml'"

[1] 2
[1] "min expr"
[1] 5.643856
[1] 3
[1] 5.643856
[1] 0.25
[1] "condition"
[1] "BH"
[1] "logical"
[1] 4
[1] 5
[1] "conditionnormal"
[1] 6
[1] 7
[1] 9
[1] 10
[1] 11
[1] 12
[1] 13
[1] 14
[1] 15
[1] 16
[1] "start"
[1] RADIL KLF1  SGCA 
Levels: KLF1 RADIL SGCA
[1] 18
# A tibble: 3 x 154
  symbol GSM109815 GSM109816 GSM109817 GSM109828 GSM109829 GSM109830 GSM109831
  <chr>      <dbl>     <dbl>     <dbl>     <dbl>     <dbl>     <dbl>     <dbl>
1 KLF1        5.47      5.60      5.39      5.42      5.40      5.43      5.57
2 RADIL       6.95      7.16      7.49      7.42      7.05      7.39      7.31
3 SGCA        4.98      5.45      5.36      5.11      5.23      5.27      5.10
# ... with 146 more variables: GSM150190 <dbl>, GSM150191 <dbl>,
#   GSM150192 <dbl>, GSM150193 <dbl>, GSM150194 <dbl>, GSM150195 <dbl>,
#   GSM150196 <dbl>, GSM150197 <dbl>, GSM150198 <dbl>, GSM150199 <dbl>,
#   GSM150201 <dbl>, GSM150202 <dbl>, GSM150203 <dbl>, GSM150204 <dbl>,
#   GSM150205 <dbl>, GSM150206 <dbl>,